# 🚀 Google Colab GPU SSH Bridge (Cloudflare Tunnel)
Этот блокнот настраивает SSH-доступ к облачному GPU Google Colab для прямого подключения с вашего локального ПК.

### Как использовать:
1. В меню выберите **Среда выполнения (Runtime)** -> **Сменить среду (Change runtime type)** -> выберите **T4 GPU** или **A100 GPU**.
2. Запустите ячейку ниже.
3. Скопируйте сгенерированную команду (вида ssh root@xxxx.trycloudflare.com) и выполните ее в терминале Windows.
4. Ключ id_ed25519 авторизуется автоматически без запроса пароля!

In [ ]:
# ==============================================================================
# Google Colab GPU -> Local SSH Bridge (Cloudflare Tunnel)
# Запустите эту ячейку в блокноте Google Colab с подключенным GPU (T4, V100, A100)
# ==============================================================================

import os
import subprocess
import time
import re

PUBLIC_KEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIAlcVVcs5DNr2kNfwLJtBjYQocNsdLCDcUKRIgquwK9J colab-bridge@local"

print("1/4. Настройка SSH сервера...")
subprocess.run("apt-get update -qq && apt-get install -qq openssh-server -y > /dev/null", shell=True)

# Авторизация SSH ключа локального ПК
ssh_dir = "/root/.ssh"
os.makedirs(ssh_dir, exist_ok=True)
os.chmod(ssh_dir, 0o700)

auth_keys = os.path.join(ssh_dir, "authorized_keys")
with open(auth_keys, "a") as f:
    f.write(PUBLIC_KEY.strip() + "\n")
os.chmod(auth_keys, 0o600)

# Конфигурация sshd
sshd_config = """
PermitRootLogin yes
PubkeyAuthentication yes
PasswordAuthentication yes
TCPKeepAlive yes
ClientAliveInterval 30
ClientAliveCountMax 10
"""
with open("/etc/ssh/sshd_config.d/colab_tunnel.conf", "w") as f:
    f.write(sshd_config)

subprocess.run("service ssh restart", shell=True)

print("2/4. Скачивание cloudflared...")
cloudflared_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_bin):
    subprocess.run(
        "curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o " + cloudflared_bin,
        shell=True,
        check=True
    )
    subprocess.run(f"chmod +x {cloudflared_bin}", shell=True, check=True)

print("3/4. Запуск туннеля к SSH (порт 22)...")
tunnel_proc = subprocess.Popen(
    [cloudflared_bin, "tunnel", "--url", "ssh://localhost:22"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_host = None
for _ in range(60):
    line = tunnel_proc.stderr.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https?://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if m:
        tunnel_host = m.group(1)
        break

if not tunnel_host:
    print("Не удалось получить URL туннеля. Проверьте вывод cloudflared:")
    print(tunnel_proc.stderr.read()[:500])
else:
    print("\n" + "="*70)
    print("  ГОТОВО! Удаленный доступ к GPU Colab открыт!")
    print("="*70)
    print("Выполните в PowerShell / cmd / терминале Windows:")
    print(f"\n  ssh root@{tunnel_host}\n")
    print("Ключ авторизуется автоматически (без ввода пароля).")
    print("В VS Code / Antigravity IDE: Remote-SSH -> Connect to Host ->")
    print(f"  root@{tunnel_host}")
    print("="*70 + "\n")

    # Предотвращаем засыпание ячейки
    while True:
        time.sleep(30)
